In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df_authors_enrich = pd.read_json("data/ETL_Centinela.raw_authors_retrieval.json")
print(f"Forma del df: {df_authors_enrich.shape}")
df_authors_enrich.head(3)

Forma del df: (65539, 4)


,_id,author_id,meta,raw
0,{'$oid': '692cca8478f12e5e6bb20861'},{'$numberLong': '10039323700'},{'retrieved_at': {'$date': '2025-11-30T22:51:4...,{'author-retrieval-response': [{'@status': 'fo...
1,{'$oid': '692cca8578f12e5e6bb20863'},{'$numberLong': '10040479200'},{'retrieved_at': {'$date': '2025-11-30T22:51:4...,{'author-retrieval-response': [{'@status': 'fo...
2,{'$oid': '692cca8778f12e5e6bb20864'},{'$numberLong': '10040712400'},{'retrieved_at': {'$date': '2025-11-30T22:51:5...,{'author-retrieval-response': [{'@status': 'fo...


In [4]:
df_temp = pd.DataFrame(
    {
        "authid": df_authors_enrich["author_id"].str["$numberLong"],
        "authors": df_authors_enrich["raw"]
        .str["author-retrieval-response"]
        .str[0]
    }
)
print(f"Forma del df: {df_temp.shape}")
df_temp.head(3)

Forma del df: (65539, 2)


,authid,authors
0,10039323700,"{'@status': 'found', '@_fa': 'true', 'coredata..."
1,10040479200,"{'@status': 'found', '@_fa': 'true', 'coredata..."
2,10040712400,"{'@status': 'found', '@_fa': 'true', 'coredata..."


In [5]:
df_authors = pd.concat(
    [
        df_temp[["authid"]].reset_index(drop=True),
        pd.DataFrame(df_temp["authors"].tolist()).reset_index(drop=True),
    ],
    axis=1,
)
print(f"Forma del df: {df_authors.shape}")
df_authors.head(3)

Forma del df: (65539, 10)


,authid,@status,@_fa,coredata,h-index,coauthor-count,affiliation-current,affiliation-history,subject-areas,author-profile
0,10039323700,found,true,{'prism:url': 'http://api.elsevier.com/content...,20,92,"{'@id': '112314961', '@href': 'http://api.else...","{'affiliation': [{'@_fa': 'true', '@id': '6002...","{'subject-area': [{'@_fa': 'true', '@abbrev': ...","{'status': 'update', 'date-created': {'@day': ..."
1,10040479200,found,true,{'prism:url': 'http://api.elsevier.com/content...,2,4,"{'@id': '114856603', '@href': 'http://api.else...",NaN,"{'subject-area': [{'@_fa': 'true', '@abbrev': ...","{'status': 'update', 'date-created': {'@day': ..."
2,10040712400,found,true,{'prism:url': 'http://api.elsevier.com/content...,11,46,"{'@id': '133222327', '@href': 'http://api.else...","{'affiliation': [{'@_fa': 'true', '@id': '1332...","{'subject-area': [{'@_fa': 'true', '@abbrev': ...","{'status': 'update', 'date-created': {'@day': ..."


In [6]:
df_authors.isnull().sum().sort_values(ascending=False)

affiliation-history    21806
coauthor-count           914
subject-areas             47
h-index                   14
@status                    0
authid                     0
@_fa                       0
coredata                   0
affiliation-current        0
author-profile             0
dtype: int64

In [7]:
df_authors = df_authors.drop(
    columns=[
        "@status",
        "coauthor-count",
        "affiliation-history",
        "@_fa",
        "coredata",
        "affiliation-current",
        "author-profile",
    ]
)

print(f"Forma del df: {df_authors.shape}")
df_authors.head(3)

Forma del df: (65539, 3)


,authid,h-index,subject-areas
0,10039323700,20,"{'subject-area': [{'@_fa': 'true', '@abbrev': ..."
1,10040479200,2,"{'subject-area': [{'@_fa': 'true', '@abbrev': ..."
2,10040712400,11,"{'subject-area': [{'@_fa': 'true', '@abbrev': ..."


In [8]:
df_authors[df_authors["subject-areas"].isnull()]

,authid,h-index,subject-areas
263,15051489600,1,None
264,15051613700,1,None
265,15051689000,1,None
1110,25823757100,1,None
1116,25923051200,0,None
1124,25928989600,0,None
1125,25929131500,0,None
1128,25929616000,0,None
1131,25934235400,1,None
1132,25935300500,1,None


In [9]:
df_exp = df_authors.copy()

df_exp["subject-areas"] = df_exp["subject-areas"].apply(
    lambda x: x.get("subject-area", []) if isinstance(x, dict) else np.nan
)

print(f"Forma del df: {df_exp.shape}")
df_exp.head(10)

Forma del df: (65539, 3)


,authid,h-index,subject-areas
0,10039323700,20,"[{'@_fa': 'true', '@abbrev': 'COMP', '@code': ..."
1,10040479200,2,"[{'@_fa': 'true', '@abbrev': 'COMP', '@code': ..."
2,10040712400,11,"[{'@_fa': 'true', '@abbrev': 'ENVI', '@code': ..."
3,10043217500,9,"[{'@_fa': 'true', '@abbrev': 'IMMU', '@code': ..."
4,10240259200,2,"[{'@_fa': 'true', '@abbrev': 'AGRI', '@code': ..."
5,10539604000,13,"[{'@_fa': 'true', '@abbrev': 'ENVI', '@code': ..."
6,10639110600,5,"[{'@_fa': 'true', '@abbrev': 'PHYS', '@code': ..."
7,10639679200,27,"[{'@_fa': 'true', '@abbrev': 'BIOC', '@code': ..."
8,10838880900,7,"[{'@_fa': 'true', '@abbrev': 'BIOC', '@code': ..."
9,10840143000,1,"[{'@_fa': 'true', '@abbrev': 'ENGI', '@code': ..."


In [10]:
df_exp = df_exp.explode("subject-areas", ignore_index=True)
print(f"Forma del df: {df_exp.shape}")
print(df_exp["subject-areas"].isnull().sum())
df_exp.head(10)

Forma del df: (379811, 3)
47


,authid,h-index,subject-areas
0,10039323700,20,"{'@_fa': 'true', '@abbrev': 'COMP', '@code': '..."
1,10039323700,20,"{'@_fa': 'true', '@abbrev': 'PHYS', '@code': '..."
2,10039323700,20,"{'@_fa': 'true', '@abbrev': 'MEDI', '@code': '..."
3,10039323700,20,"{'@_fa': 'true', '@abbrev': 'CHEM', '@code': '..."
4,10039323700,20,"{'@_fa': 'true', '@abbrev': 'MATH', '@code': '..."
5,10039323700,20,"{'@_fa': 'true', '@abbrev': 'BIOC', '@code': '..."
6,10039323700,20,"{'@_fa': 'true', '@abbrev': 'ENGI', '@code': '..."
7,10039323700,20,"{'@_fa': 'true', '@abbrev': 'MATH', '@code': '..."
8,10039323700,20,"{'@_fa': 'true', '@abbrev': 'ENGI', '@code': '..."
9,10039323700,20,"{'@_fa': 'true', '@abbrev': 'ENGI', '@code': '..."


In [11]:
subject_norm = pd.json_normalize(df_exp["subject-areas"])
print(f"Forma del df: {subject_norm.shape}")
subject_norm.head(10)

Forma del df: (379811, 4)


,@_fa,@abbrev,@code,$
0,true,COMP,1706,Computer Science Applications
1,true,PHYS,3103,Astronomy and Astrophysics
2,true,MEDI,2700,Medicine (all)
3,true,CHEM,1602,Analytical Chemistry
4,true,MATH,2611,Modeling and Simulation
5,true,BIOC,1300,"Biochemistry, Genetics and Molecular Biology (..."
6,true,ENGI,2208,Electrical and Electronic Engineering
7,true,MATH,2604,Applied Mathematics
8,true,ENGI,2213,"Safety, Risk, Reliability and Quality"
9,true,ENGI,2203,Automotive Engineering


In [12]:
subject_norm.isnull().sum().sort_values(ascending=False)

@_fa       47
@abbrev    47
@code      47
$          47
dtype: int64

In [13]:
print(f"# de areas unicas (abbrev): {subject_norm['@abbrev'].nunique()}")
print(f"# de areas unicas: {subject_norm['$'] .nunique()}")
print(subject_norm["@abbrev"].unique())
print(subject_norm["$"].unique())

# de areas unicas (abbrev): 27
# de areas unicas: 332
['COMP' 'PHYS' 'MEDI' 'CHEM' 'MATH' 'BIOC' 'ENGI' 'MATE' 'NURS' 'ENVI'
 'DECI' 'ENER' 'HEAL' 'SOCI' 'EART' 'BUSI' 'AGRI' 'ECON' 'IMMU' 'MULT'
 'ARTS' 'CENG' 'NEUR' 'PHAR' 'PSYC' 'VETE' 'DENT' nan]
['Computer Science Applications' 'Astronomy and Astrophysics'
 'Medicine (all)' 'Analytical Chemistry' 'Modeling and Simulation'
 'Biochemistry, Genetics and Molecular Biology (all)'
 'Electrical and Electronic Engineering' 'Applied Mathematics'
 'Safety, Risk, Reliability and Quality' 'Automotive Engineering'
 'Control and Optimization' 'Signal Processing' 'Engineering (all)'
 'Aerospace Engineering' 'Control and Systems Engineering'
 'Electronic, Optical and Magnetic Materials' 'LPN and LVN' 'Biochemistry'
 'Condensed Matter Physics' 'Global and Planetary Change'
 'Decision Sciences (miscellaneous)' 'Otorhinolaryngology' 'Software'
 'Computer Networks and Communications' 'Acoustics and Ultrasonics'
 'Energy (all)' 'Artificial Intelligenc

In [14]:
df_final = pd.concat(
    [
        df_exp[["authid", "h-index"]].reset_index(drop=True),
        subject_norm[["@abbrev", "$"]].reset_index(drop=True)
    ],
    axis=1
)

df_final = df_final.rename(
    columns={
        "@abbrev": "areas_abbrev",
        "$": "subject-areas"
    }
)

print(f"Forma del df_final: {df_final.shape}")
df_final.head()

Forma del df_final: (379811, 4)


,authid,h-index,areas_abbrev,subject-areas
0,10039323700,20,COMP,Computer Science Applications
1,10039323700,20,PHYS,Astronomy and Astrophysics
2,10039323700,20,MEDI,Medicine (all)
3,10039323700,20,CHEM,Analytical Chemistry
4,10039323700,20,MATH,Modeling and Simulation


In [15]:
df_final.isnull().sum().sort_values(ascending=False)

h-index          49
areas_abbrev     47
subject-areas    47
authid            0
dtype: int64

In [16]:
df_grouped = (
    df_final
    .groupby(["authid"], as_index=False)
    .agg({
        "h-index": "first",
        "areas_abbrev": list,
        "subject-areas": list
    })
)

print(f"Forma del df: {df_grouped.shape}")
df_grouped.head()

Forma del df: (65539, 4)


,authid,h-index,areas_abbrev,subject-areas
0,10039323700,20,"[COMP, PHYS, MEDI, CHEM, MATH, BIOC, ENGI, MAT...","[Computer Science Applications, Astronomy and ..."
1,10040479200,2,"[COMP, MATH, MATH, DECI, MATH]","[Computational Theory and Mathematics, Computa..."
2,10040712400,11,"[ENVI, BUSI, ENVI, AGRI, MEDI, BIOC, ENVI, AGR...","[Ecology, Business, Management and Accounting ..."
3,10043217500,9,"[IMMU, MEDI, EART, ENVI, SOCI, MEDI, MEDI, MED...","[Immunology, Medicine (all), Earth and Planeta..."
4,10240259200,2,[AGRI],[Plant Science]


In [17]:
df_grouped.drop(columns=["areas_abbrev"], inplace=True)

In [18]:
for col in ["subject-areas"]:
    df_grouped[col] = df_grouped[col].apply(
        lambda x: np.nan if isinstance(x, list) and len(x) == 1 and pd.isna(x[0]) else x
    )

In [19]:
df_grouped.isnull().sum().sort_values(ascending=False)

subject-areas    47
h-index          14
authid            0
dtype: int64

In [20]:
df_grouped["authid"] = df_grouped["authid"].astype("int64")

In [21]:
df_grouped.to_csv("data/authors_enriched.csv", index=False)

In [22]:
df_authors_ec = pd.read_csv("data/entities/autores_ecuador.csv")
df_authors_ec.head()

,authid,authname,surname,given-name,initials,orcid,afid
0,10039323700,Apolinário J.,Apolinário,J. A.,J.A.,NaN,"['101703861', '60104598']"
1,10040479200,Almeida C.,Almeida,Carlos,C.,NaN,['60104598']
2,10040712400,Carrión V.,Carrión,Víctor,V.,0009-0007-5806-5471,"['108330837', '117939271', '118823220', '13322..."
3,10043217500,Friedman R.,Friedman,Risa,R.,NaN,['60072059']
4,10240259200,Yandún S.,Yandún,Santiago,S.,NaN,['60072063']


In [23]:
df_merge = df_authors_ec.merge(
    df_grouped,
    on="authid",
    how="left"
)

print(f"Forma del df: {df_merge.shape}")
df_merge.head()

Forma del df: (65547, 9)


,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas
0,10039323700,Apolinário J.,Apolinário,J. A.,J.A.,NaN,"['101703861', '60104598']",20,"[Computer Science Applications, Astronomy and ..."
1,10040479200,Almeida C.,Almeida,Carlos,C.,NaN,['60104598'],2,"[Computational Theory and Mathematics, Computa..."
2,10040712400,Carrión V.,Carrión,Víctor,V.,0009-0007-5806-5471,"['108330837', '117939271', '118823220', '13322...",11,"[Ecology, Business, Management and Accounting ..."
3,10043217500,Friedman R.,Friedman,Risa,R.,NaN,['60072059'],9,"[Immunology, Medicine (all), Earth and Planeta..."
4,10240259200,Yandún S.,Yandún,Santiago,S.,NaN,['60072063'],2,[Plant Science]


In [24]:
df_merge.isnull().sum().sort_values(ascending=False)

orcid            41932
subject-areas       55
initials            28
given-name          28
h-index             22
authid               0
surname              0
authname             0
afid                 0
dtype: int64

In [25]:
df_merge[df_merge["authid"] == 36449349200]

,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas
1817,36449349200,Páez-Rosas D.,Páez-Rosas,Diego,D.,0000-0002-2446-9888,"['112495454', '113363380', '117356052', '12865...",20,"[Electrical and Electronic Engineering, Educat..."


In [26]:
df_merge.to_csv("data/entities/autores_ecuador_enriquecido.csv", index=False)